# connections-rl — publish model cards to the Hugging Face Hub

Regenerates all 8 model cards from the committed evaluation JSON and pushes each
one to its Hub repo as `README.md`.

**No GPU needed.** Settings → Accelerator: None, Internet: **On**.
Add-ons → Secrets → attach a secret named `HF_TOKEN` with **write** scope.
Then Save & Run All. Takes about a minute.

Safe to re-run in an existing kernel: every cell uses absolute paths and never
depends on the kernel's current directory, so re-cloning cannot pull the working
directory out from under the session.

| Repo | Scale | Stage |
|---|---|---|
| `connections-rl-sft` | 1.5B | SFT |
| `connections-rl-grpo` | 1.5B | GRPO seed 0 |
| `connections-rl-grpo-1.5b-seed1` / `-seed2` | 1.5B | GRPO replicates |
| `connections-rl-sft-7b` | 7B | SFT |
| `connections-rl-grpo-7b` | 7B | GRPO seed 0 |
| `connections-rl-grpo-7b-seed1` / `-seed2` | 7B | GRPO replicates |

Every number in every card is read from `results*/` and `results-analysis/` at
build time. The push step refuses to run if the checked-in cards differ from what
the generator produces, so the Hub cannot drift from the repo's own results.

In [ ]:
# Cell 1 — clone the repo and read the token from Kaggle Secrets
import os, subprocess, sys, shutil

WORK = '/kaggle/working'
REPO = f'{WORK}/connections-rl'

# Step out of REPO *before* deleting it. On a re-run the kernel's cwd is already
# inside REPO, and removing the directory you are standing in leaves the process
# with an unresolvable cwd ('shell-init: error retrieving current directory'),
# which breaks every relative path for the rest of the session.
os.chdir(WORK)
shutil.rmtree(REPO, ignore_errors=True)

from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')

subprocess.run(
    ['git', 'clone', '-q', 'https://github.com/jacksonmlukas/connections-rl.git', REPO],
    check=True, cwd=WORK,
)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'huggingface_hub'], check=True)

# From here on every command passes cwd=REPO explicitly rather than relying on
# the kernel's directory, so the cells are order-independent and re-runnable.
os.chdir(REPO)
print(subprocess.run(['git', 'log', '--oneline', '-1'], cwd=REPO,
                     capture_output=True, text=True).stdout.strip())

In [ ]:
# Cell 2 — regenerate the cards from the committed eval results
subprocess.run([sys.executable, 'scripts/build_model_cards.py'], check=True, cwd=REPO)

# fails loudly if generation is not reproducible
subprocess.run([sys.executable, 'scripts/build_model_cards.py', '--check'], check=True, cwd=REPO)

In [ ]:
# Cell 3 — preview one card before anything is published
from IPython.display import Markdown, display

card = open(f'{REPO}/hub_cards/connections-rl-grpo-7b.md').read()
end = card.index('---', 4) + 3
print(card[:end])                                  # YAML metadata block
display(Markdown(card[end:][:2500] + '\n\n*(truncated preview)*'))

In [ ]:
# Cell 4 — dry run: confirm which repos would be written to
subprocess.run([sys.executable, 'scripts/push_model_cards.py', '--dry-run'], check=True, cwd=REPO)

# confirm the token works and has the right identity before writing
from huggingface_hub import HfApi
api = HfApi(token=os.environ['HF_TOKEN'])
me = api.whoami()
print('authenticated as:', me['name'])
role = me.get('auth', {}).get('accessToken', {}).get('role', 'unknown')
print('token role:', role)
assert role in ('write', 'admin', 'fineGrained'), f'token needs write scope, got {role!r}'

In [ ]:
# Cell 5 — push all 8 cards
subprocess.run([sys.executable, 'scripts/push_model_cards.py'], check=True, cwd=REPO)

In [ ]:
# Cell 6 — verify: re-read each card back off the Hub and diff against local
from huggingface_hub import hf_hub_download

sys.path.insert(0, f'{REPO}/scripts')
from push_model_cards import REPOS

user = api.whoami()['name']
bad = []
for name in REPOS:
    local = open(f'{REPO}/hub_cards/{name}.md').read()
    try:
        remote = open(hf_hub_download(f'{user}/{name}', 'README.md',
                                      token=os.environ['HF_TOKEN'],
                                      force_download=True)).read()
    except Exception as exc:
        print(f'FAIL  {name}: {exc}'); bad.append(name); continue
    ok = remote.strip() == local.strip()
    print(('OK    ' if ok else 'DIFF  ') + f'https://huggingface.co/{user}/{name}')
    if not ok:
        bad.append(name)

print()
print('all cards verified live on the Hub' if not bad else f'PROBLEMS: {bad}')

In [ ]:
# Cell 7 — spot-check the defect that was live before this push:
# a 7B-only claim must never appear in a 1.5B card's own limitations section.
import glob

leaks = []
for f in sorted(glob.glob(f'{REPO}/hub_cards/*.md')):
    t = open(f).read()
    own = t[t.index('## Bias, Risks, and Limitations'):].split('**Study-level')[0]
    scale = '7B' if ('-7b' in f or f.endswith('sft-7b.md')) else '1.5B'
    leak = scale == '1.5B' and '7B' in own
    print(f"  [{scale:>4}] {os.path.basename(f):<34} {'LEAK' if leak else 'ok'}")
    if leak:
        leaks.append(f)

print()
print('no cross-scale claim leaked' if not leaks else f'LEAKS: {leaks}')